# bridgechem — a hard-sphere gas in a box

Milestone 1: an ideal-ish gas of hard spheres that fly ballistically and collide
elastically with each other and the walls. The simulation animates **live** as it
runs — the box appears and starts moving immediately, no separate `show()` step and
no HTML file.

Everything is computed in **SI units**. Lengths are entered in **nm** and masses
in **amu** for convenience; results (speeds, temperature, pressure) come back in SI.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import bridgechem as bc

# Fewer, larger particles are easiest to watch. Particles are auto-sized to fill
# ~15% of the box, and drawn at their true collision size.
system = bc.box(N=200, size=(40, 40), temperature=300)
system

## Run and watch it live

`run` integrates the system (numba-accelerated) and animates it in place. Pass
`vectors=True` to draw each particle's velocity arrow. It still returns a
`Simulation` object holding the trajectory for analysis afterwards.

Real gas speeds (hundreds of m/s) would fly past in a blur, so playback is
paced by a `speed` knob rather than shown at true speed: at the default
`speed=1` a typical particle takes a few seconds to cross the box -- slow
enough to actually follow collisions. Try `speed=0.3` (slower) or `speed=3`
(faster); it only changes the *display* pace, not the underlying physics.

In [ ]:
sim = system.run(steps=20000, vectors=True, speed=1.0)

## Speed distribution vs Maxwell–Boltzmann

The histogram of speeds should match the 2D Maxwell–Boltzmann (Rayleigh)
distribution — and the agreement improves as `N` grows.

In [ ]:
sim.histogram("speeds", frame="all")
plt.show()

## Pressure and the (2D) ideal-gas law

Pressure is measured from the momentum the particles hand to the walls. Compare it
with $P = N k_B T / A$. Larger particles read a bit high — real physics: finite
size gives an excluded area, like a van der Waals / hard-disk gas. For a nearly
ideal gas, use small particles (`radius=0.1`).

In [ ]:
dilute = bc.box(N=400, size=(60, 60), temperature=300, radius=0.1)
sim_d = dilute.run(steps=25000, animate=False)   # animate=False = run fast, no display
P, P_ideal = sim_d.calculate("pressure"), sim_d.ideal_gas_pressure()
print(f"measured P = {P:.3e} N/m")
print(f"ideal    P = {P_ideal:.3e} N/m")
print(f"ratio      = {P / P_ideal:.3f}")

## Watch a distribution *relax* to Maxwell–Boltzmann

Start every particle at the same speed (random directions). Collisions alone drive
the system to the Maxwell–Boltzmann distribution — a nice illustration of how
equilibrium emerges.

In [ ]:
system2 = bc.box(N=400, size=(60, 60), temperature=300, velocity_init="uniform_speed")
sim2 = system2.run(steps=20000)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
sim2.histogram("speeds", frame=0, ax=axes[0]); axes[0].set_title("initial (uniform speed)")
sim2.histogram("speeds", frame=-1, ax=axes[1]); axes[1].set_title("after collisions")
plt.show()

You can also replay a recorded simulation at any time with `sim.show()` (also live,
no HTML; it too takes a `speed=` argument), and step the system manually with
`system.advance()`.

## Coming next

- `system.add_interactions("LJ")` — Lennard-Jones / dispersion forces with
  velocity-Verlet, virial pressure and phase transitions.
- `system.set_temperature(...)` — gradual cooling to explore condensation.
- 3D boxes.